In [1]:
import pandas as pd
import re

In [2]:
# Full clean restart with test code removed

In [3]:
# Load data

In [4]:
import pandas as pd
import re

controls = pd.read_csv("control_library.csv", encoding="latin-1")
evidence = pd.read_csv("evidence_dataset.csv", encoding="latin-1")

controls.columns = controls.columns.str.strip().str.replace(" ", "_")
evidence.columns = evidence.columns.str.strip().str.replace(" ", "_")

evidence["Evidence_Text"] = evidence["Evidence_Text"].astype(str).str.strip()
evidence = evidence[evidence["Evidence_Text"] != ""]
evidence = evidence[evidence["Evidence_Text"].str.lower() != "nan"]
evidence = evidence.reset_index(drop=True)

# Make Control_ID match between files
controls["Control_ID"] = "C" + controls["Control_ID"].astype(str)
evidence["Control_ID"] = evidence["Control_ID"].astype(str).str.strip()

In [5]:
def split_keywords(keyword_string):
    return [k.strip().lower() for k in str(keyword_string).split(",") if k.strip()]

In [6]:
# Text Cleaning

In [7]:
def clean_text(text):
    text = str(text).lower().strip()

    # Normalize apostrophes
    text = text.replace("’", "'")

    # Remove apostrophes (so "don't" → "dont")
    text = text.replace("'", "")

    # Remove everything except letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [8]:
# Helper function for phrase matching

In [9]:
def words_near_each_other(text, required_words, max_gap=8):
    words = text.split()

    positions = {}
    for target in required_words:
        positions[target] = [i for i, word in enumerate(words) if target in word]

    if any(len(pos_list) == 0 for pos_list in positions.values()):
        return False

    all_positions = [pos for pos_list in positions.values() for pos in pos_list]

    return max(all_positions) - min(all_positions) <= max_gap

In [10]:
# Core Analyzer

In [11]:
def keyword_found(keyword, text):
    keyword = keyword.strip().lower()

    if keyword in text:
        return True

    if " " not in keyword:
        if keyword + "s" in text:
            return True

    return False


def analyze_evidence_against_control(evidence_text, control_row):
    text = clean_text(evidence_text)

    required_keywords = split_keywords(control_row["Required_Keywords"])
    optional_keywords = split_keywords(control_row["Optional_Keywords"])
    failure_keywords = split_keywords(control_row["Failure_Indicators"])

    required_matches = [kw for kw in required_keywords if keyword_found(kw, text)]
    optional_matches = [kw for kw in optional_keywords if keyword_found(kw, text)]
    failure_matches = [kw for kw in failure_keywords if keyword_found(kw, text)]

    negation_words = {"no", "not", "dont", "doesnt", "never", "without"}
    words = set(text.split())
    negation_flag = len(words.intersection(negation_words)) > 0

    # Special proximity rule for Segregation of Duties
    if control_row["Control_ID"] == "C8":
        if words_near_each_other(text, ["same", "person", "approve"], max_gap=10):
            failure_matches.append("same person approval pattern")

    score = (len(required_matches) * 2) + (len(optional_matches) * 1) - (len(failure_matches) * 3)

    if len(failure_matches) > 0:
        status = "FAIL"
    elif negation_flag:
        status = "FAIL"
    elif len(required_matches) > 0 and score >= 2:
        status = "PASS"
    else:
        status = "UNKNOWN"

    return {
        "Required_Matches": required_matches,
        "Optional_Matches": optional_matches,
        "Failure_Matches": failure_matches,
        "Negation_Flag": negation_flag,
        "Score": score,
        "Predicted_Status": status
    }

In [12]:
# Results and loop

In [13]:
results = []

for _, ev in evidence.iterrows():
    matching_control = controls[controls["Control_ID"] == ev["Control_ID"]]

    if matching_control.empty:
        print("No matching control found for:", ev["Evidence_ID"], ev["Control_ID"])
        continue

    ctrl = matching_control.iloc[0]
    result = analyze_evidence_against_control(ev["Evidence_Text"], ctrl)

    results.append({
        "Evidence_ID": ev["Evidence_ID"],
        "Control_ID": ev["Control_ID"],
        "Control_Type": ctrl["Control_Type"],
        "Domain_Group": ctrl["Domain_Group"],
        "Control_Question": ev["Control_Question"],
        "Control_Description": ctrl["Control_Description"],
        "Evidence_Text": ev["Evidence_Text"],
        "Expected_Status": ev["Expected_Status"],
        "Predicted_Status": result["Predicted_Status"],
        "Score": result["Score"],
        "Required_Matches": ", ".join(result["Required_Matches"]),
        "Optional_Matches": ", ".join(result["Optional_Matches"]),
        "Failure_Matches": ", ".join(result["Failure_Matches"]),
        "Negation_Flag": result["Negation_Flag"]
    })

results_df = pd.DataFrame(results)
results_df.head()

,Evidence_ID,Control_ID,Control_Type,Domain_Group,Control_Question,Control_Description,Evidence_Text,Expected_Status,Predicted_Status,Score,Required_Matches,Optional_Matches,Failure_Matches,Negation_Flag
0,E001,C1,Access Management,Identity & Access,Is multi-factor authentication required for us...,Multi-Factor Authentication (MFA): Requiring a...,We require MFA for all users. They log in with...,PASS,PASS,3,mfa,push notification,,False
1,E002,C1,Access Management,Identity & Access,Is multi-factor authentication required for us...,Multi-Factor Authentication (MFA): Requiring a...,"No, we don't use MFA right now. Users just log...",FAIL,FAIL,2,mfa,,,True
2,E003,C1,Access Management,Identity & Access,Is multi-factor authentication required for us...,Multi-Factor Authentication (MFA): Requiring a...,We have a standard login process in place to k...,UNKNOWN,UNKNOWN,0,,,,False
3,E004,C2,Access Management,Identity & Access,Are user access reviews performed on a defined...,Quarterly reviews to ensure terminated employe...,We perform quarterly access reviews and remove...,PASS,PASS,3,access review,terminated employee,,False
4,E005,C2,Access Management,Identity & Access,Are user access reviews performed on a defined...,Quarterly reviews to ensure terminated employe...,"We usually remove access when someone leaves, ...",FAIL,FAIL,0,,,,True


In [14]:
# Evaluate Results

In [15]:
results_df = pd.DataFrame(results)
results_df.shape

(30, 14)

In [16]:
# Accuracy

In [17]:
results_df["Expected_Status"] = results_df["Expected_Status"].astype(str).str.strip().str.upper()
results_df["Predicted_Status"] = results_df["Predicted_Status"].astype(str).str.strip().str.upper()

accuracy = (results_df["Expected_Status"] == results_df["Predicted_Status"]).mean()
print("Accuracy:", accuracy)

print(results_df.groupby(["Expected_Status", "Predicted_Status"]).size())

Accuracy: 1.0
Expected_Status  Predicted_Status
FAIL             FAIL                10
PASS             PASS                10
UNKNOWN          UNKNOWN             10
dtype: int64


In [18]:
print(evidence.columns)

Index(['Evidence_ID', 'Control_ID', 'Control_Type', 'Domain_Group',
       'Control_Question', 'Evidence_Text', 'Expected_Status'],
      dtype='object')


In [19]:
# Inspect wrong rows for accuracy

In [20]:
results_df["Correct"] = results_df["Expected_Status"] == results_df["Predicted_Status"]

wrong_results = results_df[results_df["Correct"] == False]

wrong_results[[
    "Evidence_ID",
    "Control_ID",
    "Control_Question",
    "Evidence_Text",
    "Expected_Status",
    "Predicted_Status",
    "Score",
    "Required_Matches",
    "Optional_Matches",
    "Failure_Matches",
    "Negation_Flag"
]]

,Evidence_ID,Control_ID,Control_Question,Evidence_Text,Expected_Status,Predicted_Status,Score,Required_Matches,Optional_Matches,Failure_Matches,Negation_Flag


In [21]:
# Control Level Summary Table - for 10 control view

In [22]:
control_summary = results_df[[
    "Control_ID",
    "Control_Type",
    "Domain_Group",
    "Control_Description",
    "Predicted_Status",
    "Score"
]].copy()

control_summary.head()

,Control_ID,Control_Type,Domain_Group,Control_Description,Predicted_Status,Score
0,C1,Access Management,Identity & Access,Multi-Factor Authentication (MFA): Requiring a...,PASS,3
1,C1,Access Management,Identity & Access,Multi-Factor Authentication (MFA): Requiring a...,FAIL,2
2,C1,Access Management,Identity & Access,Multi-Factor Authentication (MFA): Requiring a...,UNKNOWN,0
3,C2,Access Management,Identity & Access,Quarterly reviews to ensure terminated employe...,PASS,3
4,C2,Access Management,Identity & Access,Quarterly reviews to ensure terminated employe...,FAIL,0


In [23]:
# Collapsing controls in to 10 instead of 30

In [24]:
control_demo = results_df[results_df["Expected_Status"] == "PASS"].copy()

control_demo = control_demo[[
    "Control_ID",
    "Control_Type",
    "Domain_Group",
    "Control_Description",
    "Evidence_Text",
    "Predicted_Status",
    "Score"
]]

control_demo

,Control_ID,Control_Type,Domain_Group,Control_Description,Evidence_Text,Predicted_Status,Score
0,C1,Access Management,Identity & Access,Multi-Factor Authentication (MFA): Requiring a...,We require MFA for all users. They log in with...,PASS,3
3,C2,Access Management,Identity & Access,Quarterly reviews to ensure terminated employe...,We perform quarterly access reviews and remove...,PASS,3
6,C3,Access Management,Identity & Access,Privileged accounts (admin/root) are restricte...,Admin access is restricted to authorized perso...,PASS,3
9,C4,Access Management,Identity & Access,"Password policies enforce complexity, expirati...",Passwords must meet complexity requirements an...,PASS,4
12,C5,Change Management,Change Control,Testing & User Acceptance Testing (UAT): Ensur...,All new features go through user acceptance te...,PASS,2
15,C6,Change Management,Change Control,Change Authorization: Requiring documented app...,Every change requires a ticket and documented ...,PASS,4
18,C7,Physical Security,Infrastructure Security,"Physical Security: Keycards, biometric locks, ...",Access to data centers requires keycard entry ...,PASS,5
21,C8,Change Management,Change Control,Segregation of Duties (SoD): Ensuring the pers...,Developers cannot approve their own changes; a...,PASS,2
24,C9,Operations,IT Operations,"Automated Backups: Daily, weekly, or real-time...",We perform daily backups and regularly test da...,PASS,4
27,C10,Operations,IT Operations,Job Scheduling Monitoring: Reviewing logs to e...,We monitor batch jobs and review logs to ensur...,PASS,4


In [52]:
print("Controls:", controls.shape)
print("Evidence:", evidence.shape)

Controls: (10, 9)
Evidence: (30, 7)
